<a href="https://colab.research.google.com/github/pinkiestnnd/pemrograman-dasar/blob/main/Tugas7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nadia Putri Affanur Jabar - 4232501023 - TRPE 3A pagi - Dasar Pemrograman

***Assignment 2***

**A power plant KPI module**

Get the task from the Attachments & Downloads section below, titled Assignment 2: A Power Plant KPI Module (PDF).



In [1]:
# ==== 1. DATA UNIT 1 (1-30 November 2026) ====
energy = [5576, 6098, 6078, 6095, 6074, 6077, 5564, 5552, 6100, 6083,
          6097, 6089, 6066, 5533, 5566, 6097, 3592, 6059, 6100, 6098,
          5565, 5571, 6103, 6135, 6065, 6071, 6086, 5577, 5575, 6062]   # MWh
coal = [2998, 3258, 3250, 3263, 3249, 3246, 2993, 2983, 3258, 3258,
        3253, 3249, 3244, 2996, 3009, 3257, 1950, 3234, 3261, 3262,
        3015, 2995, 3260, 3275, 3248, 3264, 3267, 2998, 3008, 3240]    # t
run_hours = [24] * 30
run_hours[16] = 18      # hari 17: unit hanya beroperasi 18 jam

UNIT1_MW = 330          # MW terpasang
UNIT1_CV = 4400         # kcal/kg


# ==== 2. FUNGSI KPI ====
KCAL_PER_KWH_THERMAL = 860      # 1 kWh = 860 kcal
EFF_LIMIT_PCT = 36.3            # batas efisiensi minimum (%)
FULL_DAY_HOURS = 24             # jam operasi penuh per hari


def _check_energy(energy_mwh):
    """Pastikan energi positif; kalau tidak, hentikan dengan ValueError."""
    if energy_mwh <= 0:
        raise ValueError(f"Energi harus > 0 MWh, diterima: {energy_mwh}")


def heat_rate(coal_t, cv_kcal_kg, energy_mwh):
    """Hitung heat rate (kcal/kWh).

    Rumus: batubara (t) x nilai kalor (kcal/kg) / energi (MWh).
    Satuan: t x kcal/kg = 1000 kg x kcal/kg, MWh = 1000 kWh, jadi faktor 1000 saling meniadakan.
    Raise ValueError jika energi <= 0.
    """
    _check_energy(energy_mwh)
    return coal_t * cv_kcal_kg / energy_mwh


def thermal_efficiency(heat_rate_kcal_kwh):
    """Hitung efisiensi termal (%) = 860 / heat rate x 100.

    Raise ValueError jika heat rate <= 0.
    """
    if heat_rate_kcal_kwh <= 0:
        raise ValueError(f"Heat rate harus > 0, diterima: {heat_rate_kcal_kwh}")
    return KCAL_PER_KWH_THERMAL / heat_rate_kcal_kwh * 100


def capacity_factor(energy_mwh, installed_mw, hours):
    """Hitung capacity factor (%) = energi / (MW terpasang x jam) x 100.

    Raise ValueError jika energi <= 0 atau kapasitas x jam <= 0.
    """
    _check_energy(energy_mwh)
    if installed_mw * hours <= 0:
        raise ValueError("MW terpasang x jam harus > 0")
    return energy_mwh / (installed_mw * hours) * 100


def specific_coal_consumption(coal_t, energy_mwh):
    """Hitung konsumsi batubara spesifik (t/MWh) = batubara / energi.

    Raise ValueError jika energi <= 0.
    """
    _check_energy(energy_mwh)
    return coal_t / energy_mwh


def day_status(efficiency_pct, run_hours_h):
    """Tentukan status harian.

    "NOT FULL"       jika jam operasi < 24 (dicek pertama),
    "LOW EFFICIENCY" jika efisiensi < 36.3 %,
    "NORMAL"         selain itu.
    """
    if run_hours_h < FULL_DAY_HOURS:
        return "NOT FULL"
    if efficiency_pct < EFF_LIMIT_PCT:
        return "LOW EFFICIENCY"
    return "NORMAL"


# ==== 3. TES ASSERT (nilai hitungan manual) ====
def raises_value_error(func, *args):
    """Kembalikan True jika func(*args) melempar ValueError (pembantu tes)."""
    try:
        func(*args)
    except ValueError:
        return True
    return False


# heat_rate: 1 t x 4400 / 2 MWh = 2200 kcal/kWh
assert heat_rate(1, 4400, 2) == 2200
assert raises_value_error(heat_rate, 1, 4400, 0)      # energi nol
assert raises_value_error(heat_rate, 1, 4400, -5)     # energi negatif

# thermal_efficiency: 860 / 2200 x 100 = 39.0909 % ; HR 860 -> 100 %
assert abs(thermal_efficiency(2200) - 39.0909) < 1e-3
assert thermal_efficiency(860) == 100
assert raises_value_error(thermal_efficiency, 0)

# capacity_factor: 330 MW x 24 h = 7920 MWh -> 100 % ; 3960 MWh -> 50 %
assert capacity_factor(7920, 330, 24) == 100
assert capacity_factor(3960, 330, 24) == 50
assert raises_value_error(capacity_factor, 0, 330, 24)

# specific_coal_consumption: 1 t / 2 MWh = 0.5 t/MWh
assert specific_coal_consumption(1, 2) == 0.5
assert raises_value_error(specific_coal_consumption, 1, 0)

# day_status
assert day_status(37.0, 24) == "NORMAL"
assert day_status(36.0, 24) == "LOW EFFICIENCY"
assert day_status(37.0, 18) == "NOT FULL"
assert day_status(36.0, 18) == "NOT FULL"        # NOT FULL didahulukan
assert day_status(36.3, 24) == "NORMAL"          # tepat di batas = normal

print("Semua tes assert LULUS")


# ==== 4. PENGOLAHAN DATA & PROSEDUR LAPORAN ====
def build_daily_kpi(energy_mwh, coal_t, run_hours_h, installed_mw, cv_kcal_kg):
    """Hitung KPI harian untuk satu unit; kembalikan list of dict (satu dict per hari).

    Fungsi ini bisa dipakai untuk unit mana pun: cukup ganti MW terpasang dan nilai kalor.
    """
    if not (len(energy_mwh) == len(coal_t) == len(run_hours_h)):
        raise ValueError("Panjang energi, batubara, dan jam operasi harus sama")
    rows = []
    for i in range(len(energy_mwh)):
        hr = heat_rate(coal_t[i], cv_kcal_kg, energy_mwh[i])
        eff = thermal_efficiency(hr)
        rows.append({
            "day": i + 1,
            "hr": hr,
            "eff": eff,
            "cf": capacity_factor(energy_mwh[i], installed_mw, run_hours_h[i]),
            "scc": specific_coal_consumption(coal_t[i], energy_mwh[i]),
            "status": day_status(eff, run_hours_h[i]),
        })
    return rows


def print_report(rows, title):
    """Cetak tabel KPI harian (prosedur: hanya mencetak, tidak mengembalikan nilai)."""
    print(title)
    print(f"{'Day':>3} {'HR kcal/kWh':>12} {'Eff %':>7} {'CF %':>7} {'t/MWh':>7}  Status")
    print("-" * 53)
    for r in rows:
        print(f"{r['day']:>3} {r['hr']:>12,.1f} {r['eff']:>7.2f} {r['cf']:>7.1f} "
              f"{r['scc']:>7.3f}  {r['status']}")


def summarize(rows, energy_mwh, run_hours_h, installed_mw):
    """Hitung ringkasan periode; kembalikan dict.

    Kunci: avg_eff, monthly_cf (CF seluruh periode), status_count,
    worst3 (list 3 baris dengan efisiensi terendah).
    """
    avg_eff = sum(r["eff"] for r in rows) / len(rows)
    monthly_cf = capacity_factor(sum(energy_mwh), installed_mw, sum(run_hours_h))
    status_count = {"NORMAL": 0, "LOW EFFICIENCY": 0, "NOT FULL": 0}
    for r in rows:
        status_count[r["status"]] += 1
    worst3 = sorted(rows, key=lambda r: r["eff"])[:3]
    return {"avg_eff": avg_eff, "monthly_cf": monthly_cf,
            "status_count": status_count, "worst3": worst3}


def print_summary(s, title):
    """Cetak ringkasan hasil summarize() (prosedur)."""
    print(title)
    print(f"Rata-rata efisiensi     : {s['avg_eff']:.2f} %")
    print(f"Capacity factor periode : {s['monthly_cf']:.2f} %")
    print("Jumlah hari per status  :")
    for name, n in s["status_count"].items():
        print(f"   {name:<15}: {n} hari")
    print("Tiga hari paling tidak efisien:")
    for r in s["worst3"]:
        print(f"   Hari {r['day']:>2}: efisiensi {r['eff']:.2f} %  "
              f"(HR {r['hr']:,.1f} kcal/kWh)  [{r['status']}]")


# tes summarize dengan 2 hari hitungan manual
_t = build_daily_kpi([2, 2], [1, 1], [24, 24], 1, 4400)       # HR 2200, eff 39.09 %
_s = summarize(_t, [2, 2], [24, 24], 1)
assert abs(_s["avg_eff"] - 39.0909) < 1e-3
assert _s["status_count"]["NORMAL"] == 2
assert abs(_s["monthly_cf"] - 4 / 48 * 100) < 1e-9             # 4 MWh / (1 MW x 48 h)


# ==== 5. LAPORAN & RINGKASAN UNIT 1 ====
unit1_rows = build_daily_kpi(energy, coal, run_hours, UNIT1_MW, UNIT1_CV)
print_report(unit1_rows, "LAPORAN KPI UNIT 1 (330 MW, 4.400 kcal/kg) - November 2026")
print()
s1 = summarize(unit1_rows, energy, run_hours, UNIT1_MW)
print_summary(s1, "RINGKASAN UNIT 1 - November 2026")


# ==== 6. UNIT 2 (600 MW, 5.100 kcal/kg): FUNGSI YANG SAMA ====
energy2 = [12150, 12080, 11020, 12210, 12095]     # MWh
coal2 = [5240, 5225, 4790, 5260, 5230]            # t
hours2 = [24] * 5
UNIT2_MW, UNIT2_CV = 600, 5100

unit2_rows = build_daily_kpi(energy2, coal2, hours2, UNIT2_MW, UNIT2_CV)
print()
print_report(unit2_rows, "LAPORAN KPI UNIT 2 (600 MW, 5.100 kcal/kg) - 5 hari")
print()
s2 = summarize(unit2_rows, energy2, hours2, UNIT2_MW)
print_summary(s2, "RINGKASAN UNIT 2")


# ==== 7. UJI ENERGI NOL ====
print()
for bad in (0, -10):
    try:
        heat_rate(100, 4400, bad)
    except ValueError as e:
        print(f"energi = {bad:>3} -> ValueError: {e}")

Semua tes assert LULUS
LAPORAN KPI UNIT 1 (330 MW, 4.400 kcal/kg) - November 2026
Day  HR kcal/kWh   Eff %    CF %   t/MWh  Status
-----------------------------------------------------
  1      2,365.7   36.35    70.4   0.538  NORMAL
  2      2,350.8   36.58    77.0   0.534  NORMAL
  3      2,352.7   36.55    76.7   0.535  NORMAL
  4      2,355.6   36.51    77.0   0.535  NORMAL
  5      2,353.6   36.54    76.7   0.535  NORMAL
  6      2,350.2   36.59    76.7   0.534  NORMAL
  7      2,366.9   36.34    70.3   0.538  NORMAL
  8      2,364.0   36.38    70.1   0.537  NORMAL
  9      2,350.0   36.60    77.0   0.534  NORMAL
 10      2,356.6   36.49    76.8   0.536  NORMAL
 11      2,347.6   36.63    77.0   0.534  NORMAL
 12      2,347.8   36.63    76.9   0.534  NORMAL
 13      2,353.0   36.55    76.6   0.535  NORMAL
 14      2,382.5   36.10    69.9   0.541  LOW EFFICIENCY
 15      2,378.7   36.15    70.3   0.541  LOW EFFICIENCY
 16      2,350.5   36.59    77.0   0.534  NORMAL
 17      2,388.